<a href="https://colab.research.google.com/github/saatour/GW170817/blob/main/20261013_EAGLE_simulation_analysis_Chapter6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install numpy pandas matplotlib scipy astropy healpy requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 26.7 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import healpy as hp
import requests

from astropy.io import fits
from scipy.integrate import cumulative_trapezoid

# Create local data directory
DATA = Path("data")
DATA.mkdir(exist_ok=True)

# GW190814 localisation file
SKYMAP_URL = (
    "https://dcc.ligo.org/LIGO-P2000230/public/"
    "GW190814_skymap.fits.gz"
)

SKYMAP_FILE = DATA / "GW190814_skymap.fits.gz"

# Credible probability level
CREDIBLE = 0.90

print("Working directory:", Path.cwd().resolve())
print("Target credible level:", CREDIBLE)

Working directory: /content
Target credible level: 0.9


In [3]:
if not SKYMAP_FILE.exists():

    response = requests.get(
        SKYMAP_URL,
        timeout=120
    )

    response.raise_for_status()

    SKYMAP_FILE.write_bytes(response.content)

# Confirm file exists
assert SKYMAP_FILE.exists()
assert SKYMAP_FILE.stat().st_size > 1000

# Record a checksum for reproducibility
sha256 = hashlib.sha256(
    SKYMAP_FILE.read_bytes()
).hexdigest()

print("File:", SKYMAP_FILE)
print("File size:", SKYMAP_FILE.stat().st_size, "bytes")
print("SHA256:", sha256)

# Inspect the FITS structure
with fits.open(SKYMAP_FILE) as hdul:

    hdul.info()

    print("\nAvailable columns:")
    print(hdul[1].columns.names)

    print("\nHEALPix ordering:")
    print(hdul[1].header.get("ORDERING"))

File: data/GW190814_skymap.fits.gz
File size: 1472321 bytes
SHA256: 468ec8032ed11458ff9ef3c8735a039fcb0ead37f7a57be9c133f3df2dd8757e
Filename: data/GW190814_skymap.fits.gz
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1                1 BinTableHDU     32   12582912R x 4C   [D, D, D, D]   

Available columns:
['PROB', 'DISTMU', 'DISTSIGMA', 'DISTNORM']

HEALPix ordering:
NESTED


In [4]:
with fits.open(SKYMAP_FILE, memmap=False) as hdul:

    header = hdul[1].header
    table = hdul[1].data

    required_columns = {
        "PROB",
        "DISTMU",
        "DISTSIGMA",
        "DISTNORM"
    }

    available_columns = set(table.names)

    assert required_columns.issubset(available_columns), (
        f"Missing columns: "
        f"{required_columns - available_columns}"
    )

    nside = int(header["NSIDE"])
    ordering = str(header.get("ORDERING", "")).upper()

    prob = np.asarray(table["PROB"], dtype=float)
    distmu = np.asarray(table["DISTMU"], dtype=float)
    distsigma = np.asarray(table["DISTSIGMA"], dtype=float)
    distnorm = np.asarray(table["DISTNORM"], dtype=float)

# Validate prob map
assert len(prob) == hp.nside2npix(nside)
assert np.all(np.isfinite(prob))
assert np.all(prob >= 0)
assert np.isclose(prob.sum(), 1.0, atol=1e-3)

print("NSIDE:", nside)
print("Ordering:", ordering)
print("Number of pixels:", len(prob))
print("Total probability:", prob.sum())

NSIDE: 1024
Ordering: NESTED
Number of pixels: 12582912
Total probability: 0.9999999999999997
